# Challenge 1 — Cartes d’identité des CSV

Les contrats sont définis dans [config/contracts](../config/contracts/), les types dans [config/types.yaml](../config/types.yaml).
Le traitement réutilisable est dans [packages/ingestion/src/indusense_ingestion/ingestion.py](../packages/ingestion/src/indusense_ingestion/ingestion.py).
Exécuter depuis la racine du projet ou le dossier notebooks, avec l’environnement indusense.
Voir le [guide](../docs/ingestion-contracts.md) pour ajouter un dataset.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "datas").is_dir() and (ROOT.parent / "datas").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "config" / "types.yaml").is_file():
    raise FileNotFoundError("Exécuter depuis la racine du projet ou le dossier notebooks.")

from indusense_ingestion.ingestion import generate_identity


In [ ]:
reports = [
    generate_identity(
        ROOT / "datas" / "releves_incidents.csv", "releves_incidents",
        contract_path=ROOT / "config" / "contracts" / "incidents.yaml",
        types_path=ROOT / "config/types.yaml", root=ROOT,
    ),
    generate_identity(
        ROOT / "datas" / "telemetry.csv", "telemetry",
        contract_path=ROOT / "config" / "contracts" / "telemetry.yaml",
        types_path=ROOT / "config/types.yaml", root=ROOT,
    ),
]
for report in reports:
    print(f"{report['path']} — {report['rows']} lignes, {report['cols']} colonnes")
    for note in report["notes"]:
        print(f"  Observation : {note}")


# Challenge 2 — Carte d’identité SQL

Analyser le script PostgreSQL sans l’exécuter. Les données du rapport sont les tuples des INSERT ; les défauts et expressions SQL ne sont pas évalués.
Les conversions et le rendu sont dans [sql_ingestion.py](../packages/ingestion/src/indusense_ingestion/sql_ingestion.py).


In [ ]:
from indusense_ingestion.sql_ingestion import generate_sql_identity

sql_report = generate_sql_identity(ROOT / "datas" / "machine.sql", root=ROOT, types_path=ROOT / "config/types.yaml")
print(sql_report["path"])
for table_name, table in sql_report["tables"].items():
    print(f"{table_name} — {len(table['rows'])} tuples, {len(table['schema'])} colonnes déclarées")
for note in sql_report["notes"]:
    print(f"Observation : {note}")
